# Tutorial 5: Working with Detected Grid Plates

In [Tutorial 1](01_your_first_plate_image.ipynb), you saw that most arrayed colony
workflows use `GridImage`: an `Image` with row-and-column plate layout. This
tutorial picks up after detection, when that grid layout becomes biologically
useful.

Once colonies have been detected, a `GridImage` can assign each colony to a well,
extract individual wells as subimages, count colonies per grid section, and show
the detected objects together with the grid overlay.

**What you will learn:**

1. Apply a detection pipeline to a `GridImage`
2. Visualize detected colonies with grid boundaries
3. Query per-colony row, column, and section assignments
4. Extract a single well as a subimage
5. Count colonies per grid section


## Imports

In [ ]:
import phenotypic as pht
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import GaussianBlur, CLAHE
from phenotypic.detect import OtsuDetector

## Load the Grid Plate

`load_yeast_plate()` returns the same 2-by-4 `GridImage` introduced in Tutorial 1.
Here we keep the setup brief because the focus is what you can do after colonies
have been detected.


In [ ]:
plate = load_yeast_plate()
print(f"Type:    {type(plate).__name__}")
print(f"Rows:    {plate.grid.nrows}")
print(f"Columns: {plate.grid.ncols}")

## Detect Colonies First

Grid assignment needs detected objects. To keep this notebook runnable on its own,
run a compact enhance-and-detect pipeline like the ones used in the earlier
detection tutorials.


In [ ]:
pipeline = pht.ImagePipeline(
    ops=[GaussianBlur(sigma=2.0), CLAHE(clip_limit=0.01), OtsuDetector()]
)
plate = pipeline.apply(plate)
print(f"Detected colonies: {plate.num_objects}")


## Visualize the Grid Overlay

After detection, the overlay view shows the segmented colonies and the grid
boundaries at the same time. This is the quickest sanity check that colonies are
being assigned to the expected wells.


In [ ]:
plate.dash(overlay=True, show_grid=True)

The dashed lines show the grid boundaries. Each rectangular region is one grid
section, corresponding to one well on the physical plate.


## Query Grid Assignments

The `.grid.info()` method returns a DataFrame with one row per detected colony,
including its grid position: row, column, and flattened section number.


In [ ]:
info = plate.grid.info()
info.head(10)

Key columns:

- **RowNum** / **ColNum** -- grid row and column (0-indexed)
- **SectionNum** -- flattened section index (0 to nrows x ncols - 1)
- **CenterRR** / **CenterCC** -- colony centroid in pixel coordinates
- **MinRR**, **MaxRR**, **MinCC**, **MaxCC** -- bounding box


## Extract a Single Well

You can pull out any grid section as a standalone subimage using bracket indexing
on the `.grid` accessor. Let's look at the well in row 0, column 0 (top-left
corner).


In [ ]:
well = plate.grid[0, 0]
well.dash()

You can also extract an entire row or column with slicing:

```python
first_row = plate.grid[0, :]     # All 12 wells in row 0
third_col = plate.grid[:, 2]     # All 8 wells in column 2
```

## Count Colonies per Grid Section

How many colonies are in each well? `.grid.get_section_counts()` summarizes the
detected objects by grid section.


In [ ]:
counts = plate.grid.get_section_counts()
counts.head(10)

The index is the section number and the value is the colony count. Sections
with zero colonies are omitted by default.

## Summary

You now know how to use grid layout after detection:

- **`plate.dash(overlay=True, show_grid=True)`** -- visual grid overlay with detected colonies
- **`plate.grid.info()`** -- per-colony DataFrame with grid row, column, and section
- **`plate.grid[row, col]`** -- extract a single well as a subimage
- **`plate.grid.get_section_counts()`** -- colony counts per section

This is where `GridImage` becomes more than a convenient image container: each
detected colony can be traced back to the well it came from.

**Next up:** [Tutorial 6: Batch Processing](06_batch_processing.ipynb) --
process many plates at once using the command-line interface.
